In [2]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
import shap
import matplotlib.pyplot as plt

In [3]:
engine = create_engine(URL(
    account       = "UHG-UHGDWAAS",
    user          = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role          = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse     = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database      = "VING_PRD_TREND_DB",
    schema        = "TMP_1M"
))

In [30]:
df_2025 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and d.collection is not null
        and hce_admit_month between '202501' and '202512'
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

df_2026 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and d.collection is not null
        and hce_admit_month between '202601' and '202604'
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"2025: {len(df_2025)} rows | 2026: {len(df_2026)} rows")

In [ ]:
# functions kept — each called 2+ times

def mom_optimal_k(successes, trials):
    mask = (trials > 0) & (successes >= 0)
    s, n = successes[mask].values, trials[mask].values
    p = s / n
    p_bar = p.mean()
    v_obs = p.var(ddof = 1)
    n_bar = len(n) / np.sum(1.0 / n)
    v_sampling = p_bar * (1 - p_bar) / n_bar
    v_signal = v_obs - v_sampling
    if v_signal <= 0:
        return 50.0
    alpha_hat = p_bar * (p_bar * (1 - p_bar) / v_signal - 1)
    beta_hat = (1 - p_bar) * (p_bar * (1 - p_bar) / v_signal - 1)
    if alpha_hat <= 0 or beta_hat <= 0:
        return 50.0
    return alpha_hat + beta_hat


def compute_shrinkage(df, k_map):
    rate_defs = {
        "initial_adr":     ("initial_adr_cnt",    "case_count"),
        "persistent_adr":  ("persistent_adr_cnt", "case_count"),
        "p2p_overturn":    ("p2p_ovrtn_cnt",      "initial_adr_cnt"),
        "appeal_overturn": ("appeal_ovrtn_cnt",   "initial_adr_cnt"),
        "mcr_overturn":    ("mcr_ovrtn_cnt",      "initial_adr_cnt"),
        "md_review":       ("md_reviewed_cnt",    "case_count"),
    }
    if "member_appeal_cnt" in df.columns:
        rate_defs["member_appeal"] = ("member_appeal_cnt", "initial_adr_cnt")

    df_broad = df.query("case_count >= 30").copy()
    global_rates = {
        feat: df_broad[num].sum() / df_broad[denom].replace(0, np.nan).sum()
        for feat, (num, denom) in rate_defs.items()
    }

    df_out = df.query("case_count >= 30 and initial_adr_cnt >= 10").copy()

    for feat, (num, denom) in rate_defs.items():
        k_raw = k_map.get(feat, 50.0)
        k = k_raw[0] if isinstance(k_raw, tuple) else k_raw
        gr = global_rates[feat]
        df_out[f"{feat}_adj_rate"] = (df_out[num] + k * gr) / (df_out[denom] + k)

    df_out = df_out.assign(
        log_case_count = lambda x: np.log1p(x["case_count"]),
        log_initial_adr_cnt = lambda x: np.log1p(x["initial_adr_cnt"]),
    )

    rate_cols = [c for c in df_out.columns if c.endswith("_adj_rate")]
    df_out[rate_cols] = df_out[rate_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    return df_out

In [38]:
df_all = pd.concat([df_2025, df_2026], ignore_index = True).query("case_count >= 30")
df_25_broad = df_2025.query("case_count >= 30")
df_26_broad = df_2026.query("case_count >= 30")

rate_configs = {
    "initial_adr":     ("initial_adr_cnt",    "case_count"),
    "persistent_adr":  ("persistent_adr_cnt", "case_count"),
    "p2p_overturn":    ("p2p_ovrtn_cnt",      "initial_adr_cnt"),
    "appeal_overturn": ("appeal_ovrtn_cnt",   "initial_adr_cnt"),
    "mcr_overturn":    ("mcr_ovrtn_cnt",      "initial_adr_cnt"),
    "md_review":       ("md_reviewed_cnt",    "case_count"),
}

k_map = {feat: mom_optimal_k(df_all[num], df_all[denom]) for feat, (num, denom) in rate_configs.items()}
k_map["member_appeal"] = mom_optimal_k(
    df_26_broad["member_appeal_cnt"],
    df_26_broad["initial_adr_cnt"]
)

# per-year k for features with >30% divergence: mcr_overturn, appeal_overturn
k_mcr_25 = mom_optimal_k(df_25_broad["mcr_ovrtn_cnt"], df_25_broad["initial_adr_cnt"])
k_mcr_26 = mom_optimal_k(df_26_broad["mcr_ovrtn_cnt"], df_26_broad["initial_adr_cnt"])

k_appeal_25 = mom_optimal_k(df_25_broad["appeal_ovrtn_cnt"], df_25_broad["initial_adr_cnt"])
k_appeal_26 = mom_optimal_k(df_26_broad["appeal_ovrtn_cnt"], df_26_broad["initial_adr_cnt"])

# normalize: extract float if tuple (guards against validation notebook's 2-return version)
def _k_float(val):
    return val[0] if isinstance(val, tuple) else val

k_map = {feat: _k_float(v) for feat, v in k_map.items()}
k_mcr_25, k_mcr_26 = _k_float(k_mcr_25), _k_float(k_mcr_26)
k_appeal_25, k_appeal_26 = _k_float(k_appeal_25), _k_float(k_appeal_26)

print("k_map (pooled):")
for feat_name, k_val in k_map.items():
    print(f"  {feat_name:<20}: {k_val:.1f}")
print(f"\nmcr_overturn per-year: 2025 = {k_mcr_25:.1f}, 2026 = {k_mcr_26:.1f}")
print(f"appeal_overturn per-year: 2025 = {k_appeal_25:.1f}, 2026 = {k_appeal_26:.1f}")

In [40]:
iso_kpi_8 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_adj_rate", "persistent_adr_adj_rate", "p2p_overturn_adj_rate",
    "appeal_overturn_adj_rate", "mcr_overturn_adj_rate", "md_review_adj_rate",
]
iso_kpi_9 = iso_kpi_8 + ["member_appeal_adj_rate"]

# per-year k overrides for mcr_overturn and appeal_overturn
k_map_25 = {**k_map, "mcr_overturn": k_mcr_25, "appeal_overturn": k_appeal_25}
k_map_26 = {**k_map, "mcr_overturn": k_mcr_26, "appeal_overturn": k_appeal_26}

df_iso_25 = compute_shrinkage(df_2025.copy(), k_map_25)
df_iso_26 = compute_shrinkage(df_2026.copy(), k_map_26)

print(f"IF-ready: 2025 = {len(df_iso_25)} rows (8 feat) | 2026 = {len(df_iso_26)} rows (9 feat)")

In [41]:
N_BOOT = 200
BOOT_THRESHOLD_PCT = 2.5
CONSENSUS_THRESHOLD = 0.50  # majority vote — calibrated via unimodal bootstrap distribution


def bootstrap_if(X, n_boot = N_BOOT, percentile = BOOT_THRESHOLD_PCT):
    n = len(X)
    flag_counts = np.zeros(n)
    for b in range(n_boot):
        idx = np.random.choice(n, size = n, replace = True)
        X_boot = X[idx]
        model = IsolationForest(contamination = "auto", n_estimators = 300, random_state = b)
        model.fit(X_boot)
        scores = model.decision_function(X)
        threshold = np.percentile(scores, percentile)
        flag_counts += (scores < threshold).astype(int)
    return flag_counts / n_boot


np.random.seed(42)

print("Running 200x bootstrap for 2025 (8 feat)...")
df_iso_25["bootstrap_flag_pct"] = bootstrap_if(df_iso_25[iso_kpi_8].values)

print("Running 200x bootstrap for 2026 (9 feat)...")
df_iso_26["bootstrap_flag_pct"] = bootstrap_if(df_iso_26[iso_kpi_9].values)

df_iso_25 = df_iso_25.assign(flagged = lambda x: x["bootstrap_flag_pct"] >= CONSENSUS_THRESHOLD)
df_iso_26 = df_iso_26.assign(flagged = lambda x: x["bootstrap_flag_pct"] >= CONSENSUS_THRESHOLD)

print(f"\n2025 flagged: {df_iso_25['flagged'].sum()} / {len(df_iso_25)}")
print(f"2026 flagged: {df_iso_26['flagged'].sum()} / {len(df_iso_26)}")
print(f"\nConsensus threshold: {CONSENSUS_THRESHOLD} (majority vote)")

In [42]:
fig, axes = plt.subplots(1, 2, figsize = (12, 4))

for ax, df_yr, yr in [(axes[0], df_iso_25, "2025"), (axes[1], df_iso_26, "2026")]:
    ax.hist(df_yr["bootstrap_flag_pct"], bins = 30, edgecolor = "black", alpha = 0.7)
    ax.axvline(CONSENSUS_THRESHOLD, color = "red", linestyle = "--", label = f"threshold = {CONSENSUS_THRESHOLD}")
    ax.set_xlabel("Bootstrap Flag")
    ax.set_ylabel("Count")
    ax.set_title(f"{yr} — Bootstrap Flag Distribution")
    ax.legend()

plt.tight_layout()
plt.show()

In [43]:
def compute_shap(df, features):
    X = df[features].values
    model = IsolationForest(contamination = "auto", n_estimators = 300, random_state = 42)
    model.fit(X)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    df_shap = pd.DataFrame(shap_values, columns = features, index = df.index)
    df = df.assign(
        shap_driver = df_shap.apply(lambda row: row.idxmin(), axis = 1),
        shap_driver_value = df_shap.apply(lambda row: row.min(), axis = 1),
    )

    for feat in features:
        median_val = df[feat].median()
        df.loc[df["shap_driver"] == feat, "driver_direction"] = np.where(
            df.loc[df["shap_driver"] == feat, feat] > median_val, "HIGH", "LOW"
        )
    return df, shap_values, explainer


df_iso_25, shap_25, explainer_25 = compute_shap(df_iso_25, iso_kpi_8)
df_iso_26, shap_26, explainer_26 = compute_shap(df_iso_26, iso_kpi_9)

print("SHAP computed for both years.")

In [45]:
fig, axes = plt.subplots(1, 2, figsize = (14, 5))

plt.sca(axes[0])
shap.summary_plot(shap_25, df_iso_25[iso_kpi_8].values, feature_names = iso_kpi_8, show = False, plot_type = "bar")
axes[0].set_title("2025 — SHAP Feature Importance (8 feat)")

plt.sca(axes[1])
shap.summary_plot(shap_26, df_iso_26[iso_kpi_9].values, feature_names = iso_kpi_9, show = False, plot_type = "bar")
axes[1].set_title("2026 — SHAP Feature Importance (9 feat)")

plt.tight_layout()
plt.show()

In [46]:
flagged_25 = df_iso_25.query("flagged").sort_values("bootstrap_flag_pct", ascending = False)
flagged_26 = df_iso_26.query("flagged").sort_values("bootstrap_flag_pct", ascending = False)

print(f"=== 2025 FLAGGED ({len(flagged_25)}) ===")
print(flagged_25[["hospital_group", "fin_market", "case_count", "bootstrap_flag_pct",
                  "shap_driver", "driver_direction"]].to_string(index = False))

print(f"\n=== 2026 FLAGGED ({len(flagged_26)}) ===")
print(flagged_26[["hospital_group", "fin_market", "case_count", "bootstrap_flag_pct",
                  "shap_driver", "driver_direction"]].to_string(index = False))

cross_year = (
    set(flagged_25[["hospital_group", "fin_market"]].apply(tuple, axis = 1))
    & set(flagged_26[["hospital_group", "fin_market"]].apply(tuple, axis = 1))
)
print(f"\nFlagged BOTH years: {len(cross_year)}")
for h, m in sorted(cross_year):
    print(f"  {h} | {m}")

In [49]:
print("=" * 70)
print("LOC IF v7 — SUMMARY")
print("=" * 70)
print(f"""
Architecture: Separate per-year Isolation Forest (Option 3)
  2025: 8 features (excludes member_appeal) | {len(df_iso_25)} hospital-markets
  2026: 9 features (includes member_appeal) | {len(df_iso_26)} hospital-markets

Bootstrap: {N_BOOT}x resamples, consensus threshold = {CONSENSUS_THRESHOLD}
  2025 flagged: {df_iso_25['flagged'].sum()}
  2026 flagged: {df_iso_26['flagged'].sum()}
  Both years:  {len(cross_year)}

SHAP: TreeExplainer per-year model, driver = feature with most negative SHAP

Key changes from v4:
  - Separate models per year (no member_appeal imputation)
  - k_map estimated from full >=30-case population
  - Consensus threshold lowered to 0.60 (stability-calibrated)
""")

In [21]:
DRIVER_RENAME = {
    'md_review_adj_rate': 'md_escalation_rate',
    'member_appeal_adj_rate': 'member_appeal_rate',
    'persistent_adr_adj_rate': 'persistent_adr_rate',
    'initial_adr_adj_rate': 'initial_adr_rate',
}
KEEP_DRIVERS = set(DRIVER_RENAME.keys())

def build_output_table(df, year_label, features):
    df = df.copy()
    median_vals = df[features].median()
    df['driver_median'] = df['shap_driver'].map(lambda d: median_vals.get(d, np.nan))
    df['driver_actual'] = df.apply(lambda row: row.get(row['shap_driver'], np.nan), axis=1)

    out = df.query('flagged').copy()
    out = out[out['shap_driver'].isin(KEEP_DRIVERS)]
    out = out.sort_values('case_count', ascending=False)

    out['shap_driver'] = out['shap_driver'].map(DRIVER_RENAME)
    out['driver_actual_fmt'] = (out['driver_actual'] * 100).round(1).astype(str) + '%'
    out['driver_median_fmt'] = (out['driver_median'] * 100).round(1).astype(str) + '%'
    out['bootstrap_flag_pct'] = (out['bootstrap_flag_pct'] * 100).round(1)

    out = out[[
        'hospital_group', 'fin_market', 'case_count',
        'bootstrap_flag_pct', 'shap_driver',
        'driver_direction', 'driver_actual_fmt', 'driver_median_fmt'
    ]].copy()
    out.columns = ['Hospital Group', 'Market', 'Cases',
                   'Bootstrap %', 'SHAP Driver',
                   'Direction', 'Driver Value', 'Median']
    print(f"\n{'='*80}")
    print(f" {year_label} — FLAGGED HOSPITALS (sorted by case count desc)")
    print(f"{'='*80}")
    print(out.to_string(index=False))
    return out

table_25 = build_output_table(df_iso_25, '2025', iso_kpi_8)
table_26 = build_output_table(df_iso_26, '2026', iso_kpi_9)

In [53]:
DRIVER_RENAME = {
    'md_review_adj_rate': 'md_escalation_rate',
    'member_appeal_adj_rate': 'member_appeal_rate',
    'persistent_adr_adj_rate': 'persistent_adr_rate',
    'initial_adr_adj_rate': 'initial_adr_rate',
    'p2p_overturn_adj_rate' : 'p2p_overturn_rate'
}
KEEP_DRIVERS = set(DRIVER_RENAME.keys())

def build_output_table(df, year_label, features, explainer, shap_vals):
    df = df.copy()
    X = df[features].values
    model = IsolationForest(contamination='auto', n_estimators=300, random_state=42)
    model.fit(X)
    df['anomaly_score'] = model.decision_function(X)

    median_vals = df[features].median()
    df['driver_median'] = df['shap_driver'].map(lambda d: median_vals.get(d, np.nan))
    df['driver_actual'] = df.apply(lambda row: row.get(row['shap_driver'], np.nan), axis=1)

    out = df.query('flagged').copy()
    out = out[out['shap_driver'].isin(KEEP_DRIVERS)]
    out = out.sort_values('case_count', ascending=False)

    out['driver_display'] = out['shap_driver'].map(DRIVER_RENAME) + ' (' + out['driver_direction'] + ')'
    out['driver_value_fmt'] = (out['driver_actual'] * 100).round(1).astype(str) + '%'
    out['median_fmt'] = (out['driver_median'] * 100).round(1).astype(str) + '%'
    out['reldiff'] = ((out['driver_actual'] - out['driver_median']) / out['driver_median'] * 100).round(1).astype(str) + '%'
    out['anomaly_score'] = out['anomaly_score'].round(4)
    out['bootstrap_flag_pct'] = (out['bootstrap_flag_pct'] * 100).round(1)

    out = out[[
        'hospital_group', 'fin_market', 'case_count', 'anomaly_score',
        'bootstrap_flag_pct', 'driver_display',
        'driver_value_fmt', 'median_fmt', 'reldiff'
    ]].copy()
    out.insert(0, 'year', year_label)
    out.columns = ['Year', 'Hospital Group', 'Market', 'Case Count', 'Anomaly Score',
                   'Bootstrap %', 'Top Anomaly Driver',
                   'Driver Value', 'Median', 'RelDiff']
    return out

table_25 = build_output_table(df_iso_25, '2025', iso_kpi_8, explainer_25, shap_25)
table_26 = build_output_table(df_iso_26, '2026', iso_kpi_9, explainer_26, shap_26)

combined = pd.concat([table_25, table_26], ignore_index=True)
# combined.to_csv(r'C:\Users\knguy139\Documents\Projects\Data\Output\loc_if_flagged_output.csv', index=False)
# print(f"Exported {len(combined)} rows to LOC/loc_if_flagged_output.csv")
# print(combined.to_string(index=False))

combined

In [ ]:
TARGET_HOSPITAL = 'ADVOCATE HEALTH & HOSPITALS IL'

def plot_waterfall(df, features, explainer, hospital, year_label):
    explanation = explainer(df[features].values)
    explanation.feature_names = features

    mask = df['hospital_group'] == hospital
    if not mask.any():
        print(f"{hospital} not found in {year_label}")
        return

    pos = np.where(mask.values)[0][0]
    shap.plots.waterfall(explanation[pos], max_display=11, show=False)
    ax = plt.gca()

    clean_labels = []
    for lbl in ax.get_yticklabels():
        raw = lbl.get_text()
        if ' = ' in raw:
            clean_labels.append(raw.split(' = ')[1].strip())
        else:
            clean_labels.append(raw)
    ax.set_yticks(ax.get_yticks())
    ax.set_yticklabels(clean_labels)

    for patch in ax.patches:
        fc = patch.get_facecolor()
        if fc[2] > fc[0]:
            patch.set_facecolor('#002677')

    plt.title(f"{hospital} \u2014 {year_label}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_waterfall(df_iso_25, iso_kpi_8, explainer_25, TARGET_HOSPITAL, '2025')
plot_waterfall(df_iso_26, iso_kpi_9, explainer_26, TARGET_HOSPITAL, '2026')

In [ ]:
# Monthly rates for flagged hospitals (md_escalation, initial_adr, persistent_adr)
flagged_hospitals = set(
    df_iso_25.query('flagged')['hospital_group'].tolist()
    + df_iso_26.query('flagged')['hospital_group'].tolist()
)

df_monthly = pd.read_sql(f"""
    select
        d.collection as hospital_group
        , a.fin_market
        , a.hce_admit_month
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and d.collection is not null
        and hce_admit_month between '202501' and '202604'
    group by 1, 2, 3
""", engine)

df_monthly = df_monthly[df_monthly['hospital_group'].isin(flagged_hospitals)].copy()

df_monthly['md_escalation_rate'] = df_monthly['md_reviewed_cnt'] / df_monthly['case_count']
df_monthly['initial_adr_rate'] = df_monthly['initial_adr_cnt'] / df_monthly['case_count']
df_monthly['persistent_adr_rate'] = df_monthly['persistent_adr_cnt'] / df_monthly['case_count']

total_cases = df_monthly.groupby('hospital_group')['case_count'].sum().rename('total_cases')
df_monthly = df_monthly.merge(total_cases, on='hospital_group')
df_monthly = df_monthly.sort_values(['hospital_group', 'total_cases', 'hce_admit_month'], ascending=[True, False, True])

df_monthly_out = df_monthly[[
    'hospital_group', 'fin_market', 'hce_admit_month', 'case_count',
    'md_escalation_rate', 'initial_adr_rate', 'persistent_adr_rate'
]].copy()

print(f"Flagged hospitals: {len(flagged_hospitals)} | Monthly rows: {len(df_monthly_out)}")
df_monthly_out